# ModelShield Phase 2 — Advanced Adversarial AI Detection and Defense Framework

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adi-015/ModelShield/blob/phase-2/Phase2/notebooks/ModelShield_Phase2.ipynb)

ModelShield is an end-to-end adversarial defense system for deep vision classifiers (CIFAR-10).
Phase 2 implements major methodological, architectural, and evaluation enhancements over Phase 1:

1. **Multi-Attack Generation**: Full support for single-step **FGSM** and iterative **Projected Gradient Descent (PGD)** with exact $L_\infty$ projection and $[0, 1]$ physical pixel clamping.
2. **Multi-Attack Detector Mixture Training**: Solves Phase 1 single-attack noise fingerprinting by training on balanced mixtures (50% Clean, 25% FGSM, 25% PGD across varied $\epsilon$).
3. **Rigorous Data Separation**: Deterministic 40,000 / 10,000 / 10,000 split; final evaluation strictly on the complete 10,000 test set; non-shuffled validation and test DataLoaders.
4. **Validation-Based Dynamic Threshold Calibration**: Calibrates the firewall decision threshold $\tau^*$ exclusively on validation data to guarantee False Positive Rate (FPR) $\le 1.0\%$.
5. **Permanent Google Drive Checkpoint Storage**: Automatic directory management in Google Drive with file integrity verification and 4-way RNG restoration for seamless training resumption.


In [ ]:
# [Cell 1] Environment Setup & Permanent Google Drive Checkpoint Storage
import sys
import os
from pathlib import Path

# Detect execution environment
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("[*] Google Drive mounted successfully at /content/drive")
except Exception:
    IN_COLAB = False
    print("[*] Running in local/standalone environment.")

# Set up repository path
if IN_COLAB:
    repo_dir = Path("/content/ModelShield")
    if not repo_dir.exists():
        !git clone -b phase-2 https://github.com/Adi-015/ModelShield.git /content/ModelShield
    else:
        !git -C /content/ModelShield fetch origin phase-2
        !git -C /content/ModelShield checkout phase-2
        !git -C /content/ModelShield pull origin phase-2
    %cd /content/ModelShield/Phase2
    sys.path.insert(0, "/content/ModelShield/Phase2")
    sys.path.insert(0, "/content/ModelShield")
else:
    phase2_dir = Path.cwd()
    if phase2_dir.name != "Phase2":
        phase2_dir = phase2_dir / "Phase2"
    sys.path.insert(0, str(phase2_dir))
    sys.path.insert(0, str(phase2_dir.parent))

import config

# Initialize storage destinations (Permanent Drive path in Colab, local fallback)
if IN_COLAB:
    drive_base = Path("/content/drive/MyDrive/ModelShield_Phase2")

    config.setup_storage(
        use_drive=True,
        custom_root=drive_base
    )

    print(
        f"[*] Permanent Google Drive Storage configured at: "
        f"{config.STORAGE_ROOT}"
    )
else:
    config.setup_storage(use_drive=False)
    print(f"[*] Local storage initialized at: {config.STORAGE_ROOT}")

# Initialize file-based logging on Google Drive
logger = config.setup_logging()
print(f"[*] Persistent logging initialized at: {config.TRAINING_LOG_FILE}")


In [ ]:
# [Cell 2] Hardware Diagnostics & Deterministic Seed Setup
import torch
import numpy as np
import matplotlib.pyplot as plt

# Apply strict deterministic seed
config.set_seed(config.RANDOM_SEED)

device = config.DEVICE
print(f"[*] Computation Device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"[*] Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("[!] Running on CPU runtime.")


In [ ]:
# [Cell 3] CIFAR-10 Data Ingestion & Strict Split Isolation
from src.data import get_cifar10_dataloaders

# Retrieve DataLoaders with deterministic 40k/10k/10k split
train_loader, val_loader, test_loader = get_cifar10_dataloaders(
    batch_size=config.CLASSIFIER_BATCH_SIZE,
    val_split_ratio=config.VAL_SPLIT_RATIO,
    random_seed=config.RANDOM_SEED,
    num_workers=2 if torch.cuda.is_available() else 0,
    download=True
)

print(f"[*] Train batches:      {len(train_loader)} ({len(train_loader.dataset)} images)")
print(f"[*] Validation batches: {len(val_loader)} ({len(val_loader.dataset)} images) [Non-shuffled]")
print(f"[*] Untouched Test:     {len(test_loader)} ({len(test_loader.dataset)} images) [Non-shuffled]")

In [ ]:
# [Cell 4] CIFAR-10 Classifier Network Architecture
from src.models import CIFAR10Classifier, count_parameters

classifier = CIFAR10Classifier(num_classes=config.NUM_CLASSES).to(device)
print(classifier)
print(f"\n[*] Trainable Parameters: {count_parameters(classifier):,}")

In [ ]:
# [Cell 5] Resumable Classifier Training
from src.train import train_classifier, plot_training_history

# Train classifier with permanent checkpointing and automatic resumption
clf_history = train_classifier(
    model=classifier,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=config.CLASSIFIER_EPOCHS,
    lr=config.CLASSIFIER_LR,
    weight_decay=config.CLASSIFIER_WEIGHT_DECAY,
    device=device,
    resume=True
)

fig = plot_training_history(
    clf_history, 
    title="Phase 2 Baseline Classifier Training Dynamics",
    save_path=config.RESULTS_CLASSIFIER / "classifier_training_curves.png"
)
import matplotlib.pyplot as plt
plt.show()

In [ ]:
# [Cell 6] Classifier Evaluation on Complete 10,000 Test Set
from src.evaluate import evaluate_classifier

clf_test_results = evaluate_classifier(
    model=classifier,
    test_loader=test_loader,
    device=device,
    save_cm_path=config.RESULTS_CLASSIFIER / "classifier_test_confusion_matrix.png"
)

# Save complete numerical test metrics to Google Drive
clf_results_path = config.RESULTS_CLASSIFIER / "classifier_test_results.json"
config.save_json(clf_test_results, clf_results_path)

print(f"[*] Official Test Accuracy: {clf_test_results['test_accuracy']*100:.2f}%")
print(f"[*] Official Test Loss:     {clf_test_results['test_loss']:.4f}")
print(f"[*] Test metrics saved to:  {clf_results_path}")


In [ ]:
# [Cell 7] Adversarial Attacks: FGSM & Projected Gradient Descent (PGD)
from src.attacks import fgsm_attack, pgd_attack, plot_attack_visualizations
from src.data import denormalize_tensor

# Retrieve one batch of clean images from test loader
test_iter = iter(test_loader)
clean_images, clean_labels = next(test_iter)
clean_images, clean_labels = clean_images.to(device), clean_labels.to(device)

# Generate attacks at epsilon = 0.03
adv_fgsm = fgsm_attack(classifier, clean_images, clean_labels, epsilon=0.03)
adv_pgd = pgd_attack(classifier, clean_images, clean_labels, epsilon=0.03, alpha=0.005, num_iter=10)

# Verify Linf and physical bounds
orig_raw = denormalize_tensor(clean_images)
fgsm_raw = denormalize_tensor(adv_fgsm)
pgd_raw = denormalize_tensor(adv_pgd)

print(f"[*] FGSM max perturbation: {(fgsm_raw - orig_raw).abs().max().item():.4f} (<= 0.03)")
print(f"[*] PGD max perturbation:  {(pgd_raw - orig_raw).abs().max().item():.4f} (<= 0.03)")
print(f"[*] Clamping: FGSM in [{fgsm_raw.min():.2f}, {fgsm_raw.max():.2f}], PGD in [{pgd_raw.min():.2f}, {pgd_raw.max():.2f}]")

# Visualize side-by-side
fig = plot_attack_visualizations(
    clean_images=clean_images,
    adv_images=adv_pgd,
    labels=clean_labels,
    model=classifier,
    sample_indices=[0, 1, 2, 3],
    attack_name="PGD (eps=0.03, 10 steps)",
    save_path=config.RESULTS_ATTACKS / "pgd_visualizations.png"
)
plt.show()

In [ ]:
# [Cell 8] Attack Spectrum Evaluation Across Epsilon Values
from src.evaluate import evaluate_attack_spectrum

spectrum_results = evaluate_attack_spectrum(
    model=classifier,
    dataloader=test_loader,
    epsilons=config.ATTACK_EPSILONS,
    device=device,
    save_plot_path=config.RESULTS_ATTACKS / "attack_robustness_curves.png"
)

# Save numerical attack spectrum evaluation results to Google Drive
spectrum_path = config.RESULTS_ATTACKS / "spectrum_results.json"
config.save_json(spectrum_results, spectrum_path)
print(f"[*] Attack spectrum results saved to: {spectrum_path}")
plt.show()


In [ ]:
# [Cell 9] Multi-Attack Mixture Adversarial Detector Training
from src.models import AdversarialDetector, count_parameters
from src.detector import build_multi_attack_dataset
from src.train import train_detector, plot_training_history
import matplotlib.pyplot as plt

detector = AdversarialDetector(input_channels=config.IMAGE_CHANNELS).to(device)
print(f"[*] Detector Trainable Parameters: {count_parameters(detector):,}")

print("[*] Generating multi-attack mixture dataset (50% clean, 25% FGSM, 25% PGD)...")
det_train_loader = build_multi_attack_dataset(
    classifier=classifier,
    clean_loader=train_loader,
    device=device,
    shuffle=True,
    batch_size=config.DETECTOR_BATCH_SIZE,
)
det_val_loader = build_multi_attack_dataset(
    classifier=classifier,
    clean_loader=val_loader,
    device=device,
    shuffle=False,
    batch_size=config.DETECTOR_BATCH_SIZE,
)
print(f"[*] Detector Train Batches: {len(det_train_loader)} | Val Batches: {len(det_val_loader)}")

# Resumable training on balanced mixture (50% clean, 25% FGSM, 25% PGD)
det_history = train_detector(
    detector=detector,
    train_loader=det_train_loader,
    val_loader=det_val_loader,
    epochs=config.DETECTOR_EPOCHS,
    lr=config.DETECTOR_LR,
    weight_decay=config.DETECTOR_WEIGHT_DECAY,
    device=device,
    resume=True
)

fig = plot_training_history(
    det_history,
    title="Phase 2 Multi-Attack Mixture Detector Training Curves",
    save_path=config.RESULTS_DETECTOR / "detector_training_curves.png"
)
plt.show()


In [ ]:
# [Cell 10] Comprehensive 4-Facet Generalization Benchmark
from src.detector import evaluate_detector_generalization

gen_results = evaluate_detector_generalization(
    detector=detector,
    classifier=classifier,
    test_loader=test_loader,
    device=device
)

# Save 4-facet generalization benchmark metrics to Google Drive
gen_results_path = config.RESULTS_DETECTOR / "generalization_results.json"
config.save_json(gen_results, gen_results_path)

print("\n" + "="*60)
print(" DETECTOR 4-FACET GENERALIZATION BENCHMARK RESULTS")
print("="*60)
for exp_key, metrics in gen_results.items():
    print(f"\n--- {exp_key.upper()} ---")
    if "accuracy" in metrics:
        print(f"  Accuracy: {metrics['accuracy']*100:.2f}% | Recall: {metrics['recall']*100:.2f}% | FPR: {metrics['fpr']*100:.2f}% | ROC-AUC: {metrics['roc_auc']:.4f}")
    elif "test_samples" in metrics:
        print(f"  Full Test Samples: {metrics['test_samples']:,} | FPR: {metrics['clean_test_fpr']*100:.2f}% | TNR: {metrics['clean_test_tnr']*100:.2f}%")
print(f"\n[*] Generalization results saved to: {gen_results_path}")


In [ ]:
# [Cell 11] AI Firewall Dynamic Threshold Calibration (Validation Data Only)
from src.firewall import calibrate_threshold

# Calibrate threshold strictly on validation set to guarantee FPR <= 1.0%
calib_results = calibrate_threshold(
    detector=detector,
    clean_val_loader=val_loader,
    adv_val_loader=None,
    target_fpr=config.FIREWALL_CONFIG["target_fpr"],
    device=device,
    save_path=config.FIREWALL_THRESHOLD_FILE
)

optimal_threshold = calib_results["calibrated_threshold"]
print(f"[*] Calibrated Optimal Threshold: {optimal_threshold:.4f}")
print(f"[*] Target Clean FPR:            {calib_results['target_fpr']*100:.2f}%")
print(f"[*] Empirical Validation FPR:    {calib_results['empirical_val_fpr']*100:.2f}%")
print(f"[*] Calibrated threshold saved to: {config.FIREWALL_THRESHOLD_FILE}")


In [ ]:
# [Cell 12] End-to-End Firewall Security Evaluation & Latency Profiling
from src.firewall import ModelShieldFirewall, profile_firewall_latency
from src.evaluate import evaluate_firewall_defense

firewall = ModelShieldFirewall(
    classifier=classifier,
    detector=detector,
    threshold=optimal_threshold,
    device=device
)

defense_results = evaluate_firewall_defense(
    firewall=firewall,
    test_loader=test_loader,
    device=device,
    epsilon=0.03
)

# Save defense evaluation results to Google Drive
defense_path = config.RESULTS_FIREWALL / "defense_results.json"
config.save_json(defense_results, defense_path)
print(f"[*] Firewall defense results saved to: {defense_path}")

# Latency Profiling
profile = profile_firewall_latency(firewall, iterations=100)
profile_path = config.RESULTS_FIREWALL / "latency_profile.json"
config.save_json(profile, profile_path)

print(f"\n[*] Latency Profiling (100 runs):")
print(f"  - Detector Mean Latency:   {profile['detector_mean_ms']:.2f} ms")
print(f"  - Classifier Mean Latency: {profile['classifier_mean_ms']:.2f} ms")
print(f"  - Total Pipeline Latency:  {profile['total_mean_ms']:.2f} ms ({profile['throughput_fps']:.1f} FPS)")
print(f"[*] Latency profile saved to:       {profile_path}")


In [ ]:
# [Cell 13] Checkpoint Manifest & Artifact Integrity Audit
import json

manifest = {
    "framework": "ModelShield Phase 2",
    "device": str(device),
    "seed": config.RANDOM_SEED,
    "classifier_best": str(config.CLASSIFIER_CHECKPOINT_BEST),
    "classifier_best_size_mb": config.CLASSIFIER_CHECKPOINT_BEST.stat().st_size / (1024**2) if config.CLASSIFIER_CHECKPOINT_BEST.exists() else 0.0,
    "detector_best": str(config.DETECTOR_CHECKPOINT_BEST),
    "detector_best_size_mb": config.DETECTOR_CHECKPOINT_BEST.stat().st_size / (1024**2) if config.DETECTOR_CHECKPOINT_BEST.exists() else 0.0,
    "calibrated_threshold": optimal_threshold,
    "target_fpr": config.FIREWALL_CONFIG["target_fpr"],
}

manifest_path = config.STORAGE_ROOT / "phase2_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print(f"\n[*] Manifest successfully written to: {manifest_path}")
